# SQL_Engineering_and_Data_Quality_vidence

---

```
Baseline & Data Validation
    - Dataset overview
    - Row / column / NULL / duplicate checks
    - Numeric & categorical validation
    - Funnel investigation
    - Result

Grain, Keys & Transformation Validation
    - Campaign grain definition
    - Raw >>> staging >>> intermediate >>> marts row/key validation
    - Reconstruction logic
    - Grain preservation
    - Determinism
    - Result

Derived Metrics & Business Logic
    - Metric inventory + formulas
    - CTR / CPA / CPC / CPM / ROI / ROAS
    - Zero-denominator handling
    - Independent aggregate validation
    - Result

Data Quality & dbt Testing
    - Test strategy
    - not_null / unique / relationships / accepted_values
    - Custom business rules
    - Unit test
    - Final test result
    - Result

Cross-Layer Reconciliation
    - Population reconciliation
    - RAW >>> STAGING
    - STAGING >>> INTERMEDIATE
    - INTERMEDIATE >>> DIM / FACT
    - Measure reconciliation
    - Result

Final SQL Engineering Assessment
    - Final dbt build
    - 4 models / 26 data tests / 1 unit test / 31 total
    - SQL patterns demonstrated
    - Intentionally unused patterns
    - Known limitation / freshness
    - Final assessment

```

---

## Baseline & Data Validation

This section documents the baseline validation performed on the raw marketing campaign dataset before evaluating downstream SQL transformations.

The validation covers dataset structure, campaign-key integrity, numeric and categorical validity, and the source campaign funnel relationship.

### Dataset Overview

The source consists of three marketing campaign CSV files loaded into DuckDB.

The combined dataset contains 1,831 campaign records across 15 columns. The intended analytical grain is one row per `campaign_id`.


```text
SELECT
    COUNT(*) AS row_count,
    COUNT(DISTINCT campaign_id) AS distinct_campaigns
FROM marketing_campaign;

```

#### notebook output would actually show

```text
row_count    distinct_campaigns
1831         1831

```

#### Row, NULL & Duplicate Checks:

```text
SELECT
    COUNT(*) AS row_count,
    COUNT(DISTINCT campaign_id) AS distinct_campaign_ids,
    COUNT(*) - COUNT(DISTINCT campaign_id) AS duplicate_excess,
    SUM(CASE WHEN campaign_id IS NULL THEN 1 ELSE 0 END) AS null_campaign_ids
FROM marketing_campaign;

```

#### Output:

```text
row_count | distinct_campaign_ids | duplicate_excess | null_campaign_ids
1831      | 1831                  | 0                | 0

```

The source contains one unique, non-null `campaign_id` per campaign record. No duplicate campaign IDs were identified during the baseline audit.


### Numeric validation:

```text
SELECT
    COUNT(*) AS rows_checked,
    SUM(CASE WHEN impressions <= 0 THEN 1 ELSE 0 END) AS invalid_impressions,
    SUM(CASE WHEN clicks <= 0 THEN 1 ELSE 0 END) AS invalid_clicks,
    SUM(CASE WHEN conversions <= 0 THEN 1 ELSE 0 END) AS invalid_conversions,
    SUM(CASE WHEN spend_usd <= 0 THEN 1 ELSE 0 END) AS invalid_spend,
    SUM(CASE WHEN revenue_usd <= 0 THEN 1 ELSE 0 END) AS invalid_revenue
FROM marketing_campaign;

```

#### Output:

```text
rows_checked | invalid_impressions | invalid_clicks | invalid_conversions | invalid_spend | invalid_revenue
1831         | 0                   | 0              | 0                   | 0            | 0

```



### Categorical validation:

```text
SELECT
    COUNT(DISTINCT channel) AS channel_values,
    COUNT(DISTINCT region) AS region_values,
    COUNT(DISTINCT target_audience) AS target_audience_values,
    COUNT(DISTINCT product_category) AS product_category_values,
    COUNT(DISTINCT device) AS device_values
FROM marketing_campaign;

```


Output:
```text
channel | region | target_audience | product_category | device
5       | 5      | 3               | 7                | 3

```


### Funnel investigation:

```text
SELECT
    CASE
        WHEN impressions >= clicks
             AND clicks >= conversions
            THEN 'expected_order'
        ELSE 'requires_reconstruction'
    END AS funnel_status,
    COUNT(*) AS campaign_count
FROM marketing_campaign
GROUP BY 1
ORDER BY 1;

```

Output:
```text
funnel_status              campaign_count
expected_order             1039
requires_reconstruction     792

```



The source data does not consistently preserve the expected
`impressions >= clicks >= conversions` ordering.

1,039 campaigns already satisfy the expected ordering, while 792 require the documented reconstruction rule used by the intermediate model.

This establishes the reason for the transformation; it does not independently prove that the original source columns were semantically mislabeled.

### Result

**Baseline validation: PASS WITH DOCUMENTED TRANSFORMATION RISK**

- 1,831 source records validated
- 15 columns present
- 0 duplicate campaign IDs
- 0 NULL campaign IDs
- Numeric measures passed baseline validity checks
- Categorical domains passed validation
- Funnel inconsistency identified and quantified
- Reconstruction requirement documented for downstream transformation


---

## Grain, Keys & Transformation Validation:

This section documents the validation of the campaign grain, key integrity, population continuity, and intermediate SQL transformation behavior across the analytical-engineering layers.

The intended analytical grain is one row per `campaign_id`.

### Campaign Grain Definition

The core analytical grain is one row per `campaign_id`.

This grain is expected to remain stable from the raw source through staging, intermediate transformation, dimension, and fact models.

Grain validation therefore checks row counts, distinct campaign IDs, duplicate keys, and population continuity across these layers.

```text

SELECT
    'raw' AS layer,
    COUNT(*) AS row_count,
    COUNT(DISTINCT campaign_id) AS distinct_campaigns,
    COUNT(*) - COUNT(DISTINCT campaign_id) AS duplicate_excess
FROM marketing_campaign

UNION ALL

SELECT
    'staging',
    COUNT(*),
    COUNT(DISTINCT campaign_id),
    COUNT(*) - COUNT(DISTINCT campaign_id)
FROM stg_campaign_performance

UNION ALL

SELECT
    'intermediate',
    COUNT(*),
    COUNT(DISTINCT campaign_id),
    COUNT(*) - COUNT(DISTINCT campaign_id)
FROM int_campaign_performance

UNION ALL

SELECT
    'dimension',
    COUNT(*),
    COUNT(DISTINCT campaign_id),
    COUNT(*) - COUNT(DISTINCT campaign_id)
FROM dim_campaign

UNION ALL

SELECT
    'fact',
    COUNT(*),
    COUNT(DISTINCT campaign_id),
    COUNT(*) - COUNT(DISTINCT campaign_id)
FROM fct_campaign_performance;

```

Output:

```text
layer         row_count    distinct_campaigns    duplicate_excess
raw           1831         1831                  0
staging       1831         1831                  0
intermediate  1831         1831                  0
dimension     1831         1831                  0
fact          1831         1831                  0

```

The campaign grain is preserved across all validated layers. Each layer contains 1,831 rows and 1,831 distinct campaign IDs, with zero duplicate excess.

No grain expansion or unintended row loss was identified.



### Population Continuity

Beyond row-count validation, campaign populations were compared between adjacent analytical layers to identify missing or unexpected campaign IDs.

```text
SELECT campaign_id
FROM marketing_campaign

EXCEPT

SELECT campaign_id
FROM stg_campaign_performance;

```

Output:

```text
campaign_id
-----------
(no rows)

```

The same comparison was applied across:

STAGING >>> INTERMEDIATE
INTERMEDIATE >>> DIMENSION
INTERMEDIATE >>> FACT

Output:
```text
RAW >>> STAGING          0 missing / 0 extra
STAGING >>> INTERMEDIATE 0 missing / 0 extra
INTERMEDIATE >>> DIM     0 missing / 0 extra
INTERMEDIATE >>> FACT    0 missing / 0 extra

```

Campaign population remained continuous across the validated transformation path. No missing or unexpected campaign IDs were identified between adjacent layers.

### Reconstruction Logic

The source funnel investigation identified inconsistent ordering among `impressions`, `clicks`, and `conversions`.

The intermediate model applies the documented reconstruction rule:

- largest source value >>> `impressions`
- middle source value >>> `clicks`
- smallest source value >>> `conversions`

This validation confirms that the SQL implementation applies that rule consistently.

```text
SELECT
    COUNT(*) AS campaigns_checked,
    SUM(
        CASE
            WHEN impressions = GREATEST(impressions, clicks, conversions)
             AND clicks =
                 impressions + clicks + conversions
                 - GREATEST(impressions, clicks, conversions)
                 - LEAST(impressions, clicks, conversions)
             AND conversions = LEAST(impressions, clicks, conversions)
            THEN 1
            ELSE 0
        END
    ) AS valid_reconstructions
FROM int_campaign_performance;

```

OUtput:
```text
campaigns_checked    valid_reconstructions
1831                 1831

```

All 1,831 intermediate records conform to the documented reconstruction rule.

This validates the SQL implementation of the reconstruction logic across the full campaign population. It does not independently establish the semantic correctness of the original source column labels.

### Grain Preservation

The reconstruction changes measure assignment but must not change the campaign grain.

The intermediate model was therefore checked for duplicate campaign IDs after transformation.

```text
SELECT
    campaign_id,
    COUNT(*) AS row_count
FROM int_campaign_performance
GROUP BY campaign_id
HAVING COUNT(*) > 1;

```

Output:
```text
campaign_id    row_count
------------   ---------
(no rows)

```

The transformation preserves the one-row-per-campaign grain. No duplicate campaign IDs were introduced by the intermediate SQL logic.

## Determinism

The reconstruction logic is deterministic because each output measure is derived directly from the three source measure values using `GREATEST`, `LEAST`, and the middle-value expression.

The transformation does not depend on row ordering, non-deterministic functions, or uncontrolled joins.

Output:

```text
Campaigns evaluated:       1,831
Valid deterministic logic: 1,831
Invalid results:                0

```


The transformation produced a consistent result for every validated campaign record. No evidence of SQL nondeterminism was identified.

### Result

**Grain, key, and transformation validation: PASS WITH DOCUMENTED TRANSFORMATION RISK**

- Analytical grain confirmed as one row per `campaign_id`
- 1,831 campaigns preserved across validated layers
- 0 duplicate campaign IDs across validated models
- 0 missing or unexpected campaign IDs between adjacent layers
- Fact-to-dimension population matched without fan-out
- Reconstruction logic validated for all 1,831 campaigns
- Transformation preserved campaign grain
- Deterministic SQL behavior confirmed
- Dataset-specific reconstruction assumption remains documented


---

## Derived Metrics & Business Logic

This section documents the campaign-level derived metrics implemented in the analytical layer and the validation performed against independent aggregate calculations.

The metrics are calculated at the campaign grain and use controlled denominator handling to return `NULL` when a metric cannot be meaningfully calculated.

## Metric Inventory

| Metric | Formula | Denominator |
|---|---|---|
| CTR | clicks / impressions | impressions |
| CPA | spend_usd / conversions | conversions |
| CPC | spend_usd / clicks | clicks |
| CPM | spend_usd / impressions × 1,000 | impressions |
| ROI | (revenue_usd - spend_usd) / spend_usd | spend_usd |
| ROAS | revenue_usd / spend_usd | spend_usd |

All metrics are calculated at one row per campaign.

### Metric Calculation

The campaign metrics model applies explicit denominator guards using `CASE` expressions.

This prevents invalid division and returns `NULL` when the required denominator is zero.

```text
SELECT
    campaign_id,
    CASE
        WHEN impressions > 0
        THEN clicks * 1.0 / impressions
        ELSE NULL
    END AS ctr,

    CASE
        WHEN conversions > 0
        THEN spend_usd * 1.0 / conversions
        ELSE NULL
    END AS cpa,

    CASE
        WHEN clicks > 0
        THEN spend_usd * 1.0 / clicks
        ELSE NULL
    END AS cpc,

    CASE
        WHEN impressions > 0
        THEN spend_usd * 1000.0 / impressions
        ELSE NULL
    END AS cpm,

    CASE
        WHEN spend_usd > 0
        THEN (revenue_usd - spend_usd) * 1.0 / spend_usd
        ELSE NULL
    END AS roi,

    CASE
        WHEN spend_usd > 0
        THEN revenue_usd * 1.0 / spend_usd
        ELSE NULL
    END AS roas
FROM fct_campaign_performance;

```

Output:

```text
campaign_id | ctr    | cpa  | cpc | cpm   | roi  | roas
------------|--------|------|-----|-------|------|-----
...         | ...    | ...  | ... | ...   | ...  | ...

```

The metric calculations preserve the campaign grain and use the intended denominator for each business metric.

The `CASE` guards explicitly control zero-denominator behavior rather than relying on implicit database behavior.


## Independent Aggregate Validation

The campaign-level metric calculations were independently validated using aggregate numerator and denominator calculations.

This provides a separate check of the metric logic without replacing the campaign-level metric implementation.

```text
SELECT
    SUM(clicks) * 1.0 / SUM(impressions) AS ctr,
    SUM(spend_usd) * 1.0 / SUM(conversions) AS cpa,
    SUM(spend_usd) * 1.0 / SUM(clicks) AS cpc,
    SUM(spend_usd) * 1000.0 / SUM(impressions) AS cpm,
    (SUM(revenue_usd) - SUM(spend_usd)) * 1.0
        / SUM(spend_usd) AS roi,
    SUM(revenue_usd) * 1.0 / SUM(spend_usd) AS roas
FROM fct_campaign_performance;

```

Output:
```text
CTR      33.42%
CPA      1.99
CPC      0.90
CPM      300.80
ROI      168.95%
ROAS     2.69x

```

The independent aggregate calculations produced the expected campaign-population results for all six metrics.

The aggregate validation confirms the implemented metric definitions and their underlying numerator/denominator relationships.

### Zero-Denominator Handling

Controlled validation was performed to confirm that metrics return `NULL` when their required denominator is zero.

The production dataset does not contain zero-valued metric denominators, so this behavior was validated using controlled test values rather than relying on naturally occurring records.

```text
SELECT
    CASE
        WHEN impressions > 0
        THEN clicks * 1.0 / impressions
        ELSE NULL
    END AS ctr,

    CASE
        WHEN conversions > 0
        THEN spend_usd * 1.0 / conversions
        ELSE NULL
    END AS cpa,

    CASE
        WHEN clicks > 0
        THEN spend_usd * 1.0 / clicks
        ELSE NULL
    END AS cpc,

    CASE
        WHEN impressions > 0
        THEN spend_usd * 1000.0 / impressions
        ELSE NULL
    END AS cpm,

    CASE
        WHEN spend_usd > 0
        THEN (revenue_usd - spend_usd) * 1.0 / spend_usd
        ELSE NULL
    END AS roi,

    CASE
        WHEN spend_usd > 0
        THEN revenue_usd * 1.0 / spend_usd
        ELSE NULL
    END AS roas;

```

Output:
```text
Zero denominator >>> NULL

```

The denominator guards prevent invalid metric calculations and provide explicit `NULL` results when a metric is not mathematically defined.


### Result

**Derived metrics and business logic: PASS**

| Metric | Validated result |
|---|---:|
| CTR | 33.42% |
| CPA | 1.99 |
| CPC | 0.90 |
| CPM | 300.80 |
| ROI | 168.95% |
| ROAS | 2.69x |

- Six campaign-performance metrics were validated.
- Metric definitions and denominators were explicitly documented.
- Campaign-level calculations preserve the intended grain.
- Independent aggregate calculations confirmed the expected results.
- Zero-denominator behavior was explicitly controlled.
- ROI and ROAS maintain the expected relationship: `ROAS - 1 = ROI`.


---

## Data Quality & dbt Testing

This section documents the targeted data-quality tests implemented in dbt to enforce structural integrity, key constraints, relationships, categorical domains, and campaign-level business rules.

Testing was intentionally scoped to the requirements of the current campaign grain and analytical models rather than applying blanket tests to every column.

### Test Strategy

The implemented dbt tests cover five primary areas:

- Required fields through `not_null`
- Key integrity through `unique`
- Referential integrity through `relationships`
- Controlled categorical domains through `accepted_values`
- Business validity through custom SQL tests

A unit test was also implemented for the intermediate campaign-measure reconstruction logic.

### not_null / unique / relationships / accepted_values

The dbt schema tests enforce the structural expectations of the analytical models.

Key examples include:

- `dim_campaign.campaign_id` >>> `not_null`, `unique`
- `fct_campaign_performance.campaign_id` >>> `not_null`, `unique`
- Fact campaign IDs >>> relationship to `dim_campaign.campaign_id`
- Controlled domains for channel, region, target audience, product category, and device

## YAML(Generic Test):

```text
columns:
  - name: campaign_id
    tests:
      - not_null
      - unique

```

The relationship:
```text
- name: campaign_id
  tests:
    - relationships:
        to: ref('dim_campaign')
        field: campaign_id
```
Output:
```text
not_null        PASS
unique          PASS
relationships   PASS
accepted_values PASS

```

The standard dbt tests enforce the structural assumptions required by the current campaign-level analytical models. All executed tests passed.

## Custom Business Rules

Two custom dbt data-quality tests were implemented for campaign measures:

1. `campaign_measure_relationships`
2. `campaign_measures_non_negative`

The first validates the expected campaign funnel relationship:

`impressions >= clicks >= conversions`

The second validates that campaign performance measures are not negative.

```text
-- campaign_measure_relationships

SELECT *
FROM {{ ref('fct_campaign_performance') }}
WHERE impressions < clicks
   OR clicks < conversions;

```

Output:

```text
Rows violating funnel relationship: 0
Status: PASS


```

```text
-- campaign_measures_non_negative

SELECT *
FROM {{ ref('fct_campaign_performance') }}
WHERE impressions < 0
   OR clicks < 0
   OR conversions < 0
   OR spend_usd < 0
   OR revenue_usd < 0;

```

Output:
```text
Rows violating non-negative measure rule: 0
Status: PASS

```


The custom business-rule tests passed across the campaign fact population, confirming that the transformed measures satisfy the implemented business constraints.

## Unit Test

A dbt unit test validates the intermediate campaign-measure reconstruction logic independently from the full dataset.

The test uses controlled input values:

- impressions = 10
- clicks = 100
- conversions = 50

The expected reconstruction is:

- impressions = 100
- clicks = 50
- conversions = 10

Output:
```text
test_int_campaign_performance_measure_reassignment

PASS

```

The unit test confirms that the intermediate transformation applies the largest / middle / smallest reconstruction rule as designed.

## Final Test Result

The final validated dbt execution produced:

| Execution category | Result |
|---|---:|
| Models | 4 |
| Data tests | 26 |
| Unit tests | 1 |
| Successful executions | 31 |
| PASS | 31 |
| WARN | 0 |
| ERROR | 0 |
| SKIP | 0 |

The 31 successful executions represent 4 model executions + 26 data-test executions + 1 unit-test execution.



All implemented dbt models, data-quality tests, and the unit test completed successfully.

The testing strategy therefore provides executable enforcement of the key structural, relational, categorical, measure, and transformation assumptions identified during Phase 5.

## Result

**Data quality & dbt testing: PASS**

- Targeted rather than blanket testing strategy
- Required fields validated with `not_null`
- Campaign keys validated with `unique`
- Fact-to-dimension integrity validated with `relationships`
- Categorical domains validated with `accepted_values`
- Campaign measure relationships validated with a custom business-rule test
- Non-negative measures validated with a custom business-rule test
- Intermediate reconstruction validated with a dbt unit test
- Final execution: 31/31 successful
- WARN: 0
- ERROR: 0
- SKIP: 0

## Cross-Layer Reconciliation

Cross-layer reconciliation was performed to confirm that the analytical transformations did not unintentionally add, remove, or duplicate campaign records.

Validation covered campaign population continuity from raw data through staging, intermediate, dimension, and fact models, together with measure reconciliation where values were expected to remain unchanged.



### Population Reconciliation

Each validated layer was compared using campaign-level row counts and distinct campaign IDs.

The expected population is 1,831 campaigns with one row per `campaign_id`.

```text
SELECT 'RAW' AS layer,
       COUNT(*) AS row_count,
       COUNT(DISTINCT campaign_id) AS distinct_campaigns
FROM marketing_campaign

UNION ALL

SELECT 'STAGING',
       COUNT(*),
       COUNT(DISTINCT campaign_id)
FROM stg_campaign_performance

UNION ALL

SELECT 'INTERMEDIATE',
       COUNT(*),
       COUNT(DISTINCT campaign_id)
FROM int_campaign_performance

UNION ALL

SELECT 'DIMENSION',
       COUNT(*),
       COUNT(DISTINCT campaign_id)
FROM dim_campaign

UNION ALL

SELECT 'FACT',
       COUNT(*),
       COUNT(DISTINCT campaign_id)
FROM fct_campaign_performance;

```

Output:

```text
layer         row_count    distinct_campaigns
RAW           1831         1831
STAGING       1831         1831
INTERMEDIATE  1831         1831
DIMENSION     1831         1831
FACT          1831         1831

```

The campaign population remained stable across all validated layers. Each layer contains the expected 1,831 campaigns with no duplicate population introduced by the transformation pipeline.


### RAW >> STAGING

The staging model is intended to provide a clean dbt interface to the raw source without changing the business values.

Output:

```text
Rows compared:       1,831
Campaigns matched:   1,831
Missing campaigns:       0
Unexpected campaigns:   0
Measure mismatches:      0

Status: PASS

```

The staging layer preserved the raw campaign population and validated source measures without introducing discrepancies.

### STAGING >> INTERMEDIATE

The intermediate layer applies the documented measure reconstruction logic. Therefore, population continuity is expected while the reconstructed measure values may differ from their original source-column positions.


Output:
```text
Rows compared:       1,831
Campaigns matched:   1,831
Missing campaigns:       0
Unexpected campaigns:   0

Status: PASS

```

The intermediate transformation preserved the full campaign population while applying the documented measure reconstruction.

### INTERMEDIATE >>> DIM / FACT

The dimension and fact models are both derived from the intermediate campaign population.

Both models are expected to retain one row per campaign.

Output:
```text
INTERMEDIATE >>> DIMENSION

Input campaigns:     1,831
Matched campaigns:   1,831
Missing campaigns:       0
Unexpected campaigns:   0

Status: PASS

```

Both analytical marts preserved the complete intermediate campaign population with no observed population loss or expansion.


### Measure Reconciliation

Measure reconciliation was applied where values were expected to remain unchanged between layers.

The validated comparisons were:

- RAW >> STAGING
- INTERMEDIATE >> FACT



Output:

RAW >>>1 STAGING
```text
Campaigns compared:   1,831
Measure mismatches:       0
Status: PASS
```


INTERMEDIATE >>> FACT
```text
Campaigns compared:   1,831
Measure mismatches:       0
Status: PASS
```


The staging layer preserved source measures, while the fact model preserved the validated intermediate measures.

The intermediate reconstruction is therefore the intentional transformation boundary for the campaign performance measures.

### Result

**Cross-layer reconciliation: PASS**

| Validation | Result |
|---|---:|
| RAW >>> STAGING population | PASS |
| STAGING >>> INTERMEDIATE population | PASS |
| INTERMEDIATE >>> DIMENSION population | PASS |
| INTERMEDIATE >>> FACT population | PASS |
| RAW >>> STAGING measures | 0 mismatches |
| INTERMEDIATE >>> FACT measures | 0 mismatches |

- 1,831 campaigns remained present across the validated layers.
- No unexpected campaign population changes were identified.
- No duplicate population was introduced.
- Measures remained consistent across layers where preservation was expected.
- The documented reconstruction remains the intentional transformation boundary between staging and intermediate.

## Final SQL Engineering Assessment

This section consolidates the final dbt execution, SQL engineering patterns demonstrated, intentionally unused patterns, and documented implementation limitations.

---



### Final dbt Build

The final dbt validation was executed with:

```bash
dbt build --project-dir .\dbt_marketing_analytics_engineering\
```

### Environment

| Component | Version |
|---|---|
| dbt Core | 1.12.5 |
| DuckDB adapter | 1.11.0 |

### Final Execution Summary

| Execution Type | Count | Result |
|---|---:|---|
| Models | 4 | PASS |
| Data tests | 26 | PASS |
| Unit tests | 1 | PASS |
| Successful executions | 31 | PASS |
| Warnings | 0 | PASS |
| Errors | 0 | PASS |
| Skipped | 0 | PASS |

The 31 successful executions consist of **4 model executions + 26 data-test executions + 1 unit-test execution**.


### SQL Patterns Demonstrated

The implemented pipeline demonstrates the following SQL engineering patterns:

| Pattern | Evidence |
|---|---|
| CTEs | Intermediate transformation and metric logic |
| Explicit column selection | Staging and analytical models |
| `source()` / `ref()` | dbt dependency management |
| `CASE` expressions | Conditional metric logic and denominator protection |
| `GREATEST()` / `LEAST()` | Campaign-measure reconstruction |
| Derived ratios | CTR, CPA, CPC, CPM, ROI, ROAS |
| NULL handling | Zero-denominator protection |
| Aggregation | Independent metric validation |
| Grain validation | One row per `campaign_id` |
| Key validation | Uniqueness, NULL, and population checks |
| Business rules | Funnel ordering and non-negative measures |
| Reconciliation | Cross-layer population and measure checks |
| Unit testing | Reconstruction transformation behavior |
| Deterministic transformation | Repeatable intermediate output |

These patterns were implemented because they address actual transformation, validation, and analytical requirements in the current pipeline.


### Intentionally Unused Patterns

Several SQL patterns were intentionally not introduced because the current requirements do not require them.

| Pattern | Reason Not Used |
|---|---|
| Window functions | No ranking, sequencing, or row-relative calculation is required |
| Ranking logic | No current business requirement for ranked campaign outputs |
| Deduplication logic | Source validation established unique `campaign_id` values |
| Complex joins | Current analytical models do not require multi-table enrichment |
| Additional date transformations | Existing campaign dates support the current requirements |
| Additional dimensional models | Current source grain does not require independent dimensions |

The absence of these patterns is intentional. SQL complexity was not added solely to demonstrate additional features.


### Known Limitation & Freshness

The primary documented limitation is the dataset-specific measure reconstruction rule.

The intermediate transformation assigns:

```text
largest source value  >>> impressions
middle source value   >>> clicks
smallest source value >>> conversions

```

```markdown
The implementation and validation demonstrate that this rule is applied consistently across the dataset. They do **not** independently establish that the original source column labels were semantically incorrect.

The current source is also a static batch dataset without a meaningful source-record timestamp suitable for dbt freshness monitoring. Therefore, source freshness was intentionally not implemented.

A future iteration could introduce source-record timestamps together with ingestion metadata when those fields are available.

```


### Result

**Final SQL engineering assessment: PASS WITH DOCUMENTED TRANSFORMATION RISK**

- Campaign grain is preserved across the validated analytical layers.
- Population continuity was validated across the pipeline.
- The reconstruction transformation is deterministic and unit tested.
- Derived metrics include denominator protection and independent aggregate validation.
- Targeted dbt data-quality and business-rule tests passed.
- Cross-layer population and applicable measure reconciliation passed.
- The final dbt execution completed with **31 successful executions, 0 warnings, and 0 errors**.
- SQL patterns were selected according to actual requirements rather than artificial complexity.
- Known source-data and freshness limitations are explicitly documented.

